# Label compartments by eye

A tool for building labels where none exist, such as primary versus terminal axon. It pairs ossify's per-vertex features with the `VertexExplorer` widget in `vertex_explorer/`:

- **Left:** a UMAP of one cell's skeleton vertices in feature space.
- **Right:** the same cell in 3D, with its output synapses.
- **Lasso** points on the left and they light up on the cell. Trim the selection to the longest path to the root, then mark it. Labels are saved per cell as you go.

The saved labels then train a classifier with the same feature sets the bundled models use, so the result can be wrapped in a `StructuredLabeler` and decoded like any other.

Run from this folder with the extra packages the widget needs:

```bash
uv run --with anywidget --with umap-learn --with scikit-learn --with xgboost --with pyarrow --with jupyterlab jupyter lab
```

In [ ]:
from pathlib import Path

DATASET = "v1dd"   # or "minnie65"
SOURCES = {
    "minnie65": dict(datastack="minnie65_public", version=1822, cells="gs://csm-skel/osy/minnie65/skel"),
    "v1dd": dict(datastack="v1dd", version=1643, cells="gs://csm-skel/osy/v1dd/skel"),
}
SOURCE = SOURCES[DATASET]

FEATURE_SET = "local"                                 # a name in ossify.compartments.FEATURE_SETS
LABEL_DIR = Path("labels") / "axon_parts" / DATASET   # one parquet per cell: features plus a label column
HUE = "down30um_syn_out_per_um"                       # what colours the points to begin with

In [ ]:
import logging
import warnings

import numpy as np
import pandas as pd
import umap
from caveclient import CAVEclient
from cloudfiles import CloudFiles
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import ossify
from ossify.compartments import FEATURE_SETS, make_skel_prop_df
from ossify.translate import SWCCompartment
from vertex_explorer import VertexExplorer, read_labels

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)

cell_files = ossify.CellFiles(SOURCE["cells"])
spec, windows = FEATURE_SETS[FEATURE_SET]

## 1. Cells

Proofread cells that are in the bucket. Cells that already have saved labels come first, so earlier work is the first thing you see.

In [ ]:
client = CAVEclient(SOURCE["datastack"])
proof = client.materialize.live_query(
    "proofreading_status_and_strategy", timestamp=client.materialize.get_timestamp(SOURCE["version"])
)
proof = proof[proof.status_axon & proof.status_dendrite].drop_duplicates("pt_root_id")
proofread = proof.pt_root_id.astype("int64").to_numpy()
in_bucket = CloudFiles(SOURCE["cells"]).exists([f"{r}.osy" for r in proofread])
available = [int(r) for r in proofread if in_bucket[f"{r}.osy"]]

labelled = sorted(int(p.stem) for p in LABEL_DIR.glob("*.parquet"))
root_ids = labelled + [r for r in available if r not in set(labelled)]
print(len(available), "cells available;", len(labelled), "already have labels")

## 2. Features and embedding

`prepare` turns a root id into what the widget needs: the axon alone, its per-vertex features with a UMAP of them, and the whole cell to draw faintly behind. Change the mask to label a different part of the cell.

In [ ]:
def features_of(part):
    """Per-vertex features of a (masked) cell, indexed by skeleton vertex id."""
    df = make_skel_prop_df(part, downstream_hops=0, feature_spec=spec, windows=windows)
    df["dist_to_root_um"] = np.asarray(part.skeleton.distance_to_root()) / 1000
    return df

FEATURES = None  # set from the first cell: every column the feature set makes

def prepare(root_id):
    """root id -> (axon, features with umap0/umap1, whole cell)."""
    global FEATURES
    cell = cell_files.load(f"{root_id}.osy")
    is_axon = np.asarray(cell.skeleton.get_feature("compartment")) == int(SWCCompartment.AXON)
    axon = cell.apply_mask(mask=is_axon, layer="skeleton")
    df = features_of(axon)
    if FEATURES is None:
        FEATURES = [c for c in df.columns if c != "dist_to_root_um"]
    X = np.log1p(df[FEATURES].clip(lower=0).fillna(0).to_numpy(dtype=float))  # counts and densities are skewed
    xy = make_pipeline(StandardScaler(), umap.UMAP(random_state=0)).fit_transform(X)
    df["umap0"], df["umap1"] = xy[:, 0], xy[:, 1]
    return axon, df, cell

## 3. Label

- **Lasso** on the left; shift adds to the selection, alt removes, a click picks one vertex.
- **Longest path to root** trims the selection to a single trunk.
- **Label primary / Label terminal** marks the selection; **Unlabel** clears it.
- The arrows step through the cells. Each label change writes `labels/axon_parts/<dataset>/<root id>.parquet`.
- On the cell: drag rotates, shift-drag pans, scroll zooms, double-click returns to the front view.

The selection is live in Python: `vx.selected_vertices`, `vx.selected_frame()`.

In [ ]:
vx = VertexExplorer.from_loader(prepare, root_ids, start=0, hue=HUE, save_dir=LABEL_DIR)
vx

## 4. Labels so far

In [ ]:
labels = read_labels(LABEL_DIR)[["root_id", "vertex_id", "label"]]
labels.groupby(["root_id", "label"]).size().unstack(fill_value=0)

## 5. A first classifier

Features are recomputed with the current feature set, so labels made with older features still count. Each cell is predicted by a model that did not see it.

In [ ]:
import xgboost

def labelled_features(root_id):
    _, df, _ = prepare(root_id)
    mine = labels[labels.root_id == root_id].set_index("vertex_id").label
    return df.assign(label=mine.reindex(df.index).to_numpy(), root_id=root_id)

frames = pd.concat([labelled_features(r) for r in labels.root_id.unique()])
train = frames.dropna(subset=["label"])
classes = sorted(train.label.unique())
y = train.label.map({name: i for i, name in enumerate(classes)}).to_numpy()

def fit(rows, target):
    return xgboost.XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=4, subsample=0.8).fit(
        rows[FEATURES].to_numpy(dtype=float), target
    )

rows = []
for held in train.root_id.unique():
    inside = (train.root_id != held).to_numpy()
    model = fit(train[inside], y[inside])
    predicted = model.predict(train[~inside][FEATURES].to_numpy(dtype=float))
    for i, name in enumerate(classes):
        is_class = y[~inside] == i
        rows.append({"held-out cell": held, "label": name, "vertices": int(is_class.sum()), "correct": float((predicted[is_class] == i).mean()) if is_class.any() else np.nan})
pd.DataFrame(rows).pivot(index="held-out cell", columns="label", values=["vertices", "correct"]).round(3)

## 6. Check the classifier on the cell

Colour the points by the classifier's call to see where it disagrees with your eye, and correct those. Repeat from step 4.

In [ ]:
model = fit(train, y)

def prepare_with_prediction(root_id):
    axon, df, cell = prepare(root_id)
    df[f"p_{classes[0]}"] = model.predict_proba(df[FEATURES].to_numpy(dtype=float))[:, 0]
    return axon, df, cell

vx = VertexExplorer.from_loader(prepare_with_prediction, root_ids, start=0, hue=f"p_{classes[0]}", save_dir=LABEL_DIR)
vx